In [53]:
from google.cloud import bigquery
from google.oauth2 import service_account

import psycopg2
import pandas as pd
from sqlalchemy import create_engine,URL

Matplotlib is building the font cache; this may take a moment.


In [3]:
credentials = service_account.Credentials.from_service_account_file(
  'c:/Users/Bob/oasisbiz/datawarehouse-390004-34bcb00fb7cb.json'
)
project_id = 'datawarehouse-390004'

In [4]:
client = bigquery.Client(
  project=project_id,
  credentials=credentials
)

In [7]:
# job_config = bigquery.LoadJobConfig(
#   schema=[
#     bigquery.SchemaField(name='poi_index',field_type='NUMERIC'),
#     bigquery.SchemaField(name='lat',field_type='NUMERIC'),
#     bigquery.SchemaField(name='lon',field_type='NUMERIC')
#   ],
#   create_disposition="CREATE_IF_NEEDED",
#   write_disposition="WRITE_APPEND",
#   source_format=bigquery.SourceFormat.CSV,
#   field_delimiter='|'
# )

In [95]:
# # load table from file
# poi_df.to_csv(
#   'poi_df.csv',
#   header=False,
#   index=False,
#   sep='|'
# )
# load_job = client.load_table_from_file(
#   open('poi_df.csv','rb'),
#   'temp.poi_df',
#   job_config=job_config
# )
# load_job.result()

In [96]:
job = client.query(
  f'''
  select
    deal_amount,
    contract_date,
    floor_ floor,
    plottage,
    pnu,
    st_x(trade_case_point) longitude,
    st_y(trade_case_point) latitude,
    building_type,
    building_detail_type,
    building_use,
    build_year,
    building_area,
    sig_cd,
    emd_cd,
    land_use
  from m2.cremao_real_estate_trade_case
  where
    sido_cd = '11' and
    building_type = '집합' and
    contract_date >= '2020-01-01'
  '''
)
deal_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [97]:
# create columns from contract_date
deal_df['contract_year'] = [
  date.year
  for date
  in deal_df['contract_date']
]
deal_df['contract_month'] = [
  date.month
  for date
  in deal_df['contract_date']
]
deal_df['contract_day'] = [
  date.day
  for date
  in deal_df['contract_date']
]

In [98]:
deal_df['building_age'] = deal_df['contract_year'] - deal_df['build_year'].astype('float')

In [99]:
deal_df['geom'] = 'POINT (' + deal_df['longitude'].astype('string') + ' ' + deal_df['latitude'].astype('string') + ')'

In [100]:
deal_df = deal_df[[
  'deal_amount','contract_year','contract_month','contract_day','floor','plottage','land_use','build_year','building_age','building_type','building_detail_type','building_use','building_area','sig_cd','emd_cd','pnu','longitude','latitude','geom'
]]

In [101]:
deal_df_clean = deal_df[0:0]

for sig in deal_df['sig_cd'].unique():
  tmp_df = deal_df[deal_df['sig_cd'] == sig]
  # IQR 계산
  Q1 = tmp_df['deal_amount'].quantile(0.25)
  Q3 = tmp_df['deal_amount'].quantile(0.75)
  IQR = Q3 - Q1

  # # 이상치 제거
  deal_df_clean = pd.concat([
    deal_df_clean,
    tmp_df[~((tmp_df['deal_amount'] < (Q1 - 1.5 * IQR)) | (tmp_df['deal_amount'] > (Q3 + 1.5 * IQR)))]
  ])

In [103]:
print(len(deal_df_clean), '/', len(deal_df))

11668 / 12664


In [88]:
# connect to localhost
conn = psycopg2.connect(
  host='localhost',
  port=5432,
  database='postgres',
  user='postgres',
  password='postgres'
)
conn.set_session(autocommit=True)
cursor = conn.cursor()

In [104]:
try:
  cursor.execute(
    f'''
    create table m1_y (
      deal_amount numeric,
      contract_year numeric,
      contract_month numeric,
      contract_day numeric,
      floor numeric,
      plottage numeric,
      land_use varchar,
      build_year numeric,
      building_age numeric,
      building_type varchar,
      building_detail_type varchar,
      building_use varchar,
      building_area numeric,
      sig_cd varchar,
      emd_cd varchar,
      pnu varchar,
      longitude varchar,
      latitude varchar,
      geom geometry(geometry,4326)
    )
    '''
  )
except Exception as err:
  print(err)

오류:  "m1_y" 이름의 릴레이션(relation)이 이미 있습니다



In [39]:
engine = create_engine(
  URL.create(
    drivername='postgresql+psycopg2',
    host='localhost',
    port=5432,
    database='postgres',
    username='postgres',
    password='postgres'
  )
)

In [105]:
try:
  cursor.execute(
    'delete from m1_y'
  )
  deal_df_clean.to_sql(
    'm1_y',
    engine,
    if_exists='append',
    index=False,
  )
except Exception as err:
  print(err)

In [107]:
deal_df_clean[
  ['deal_amount','contract_year','contract_month','contract_day','floor','land_use','build_year','building_age','building_detail_type','building_use','building_area','sig_cd','emd_cd','pnu','longitude','latitude']
].to_csv(
  'm1_data_06.11.csv',
  sep=',',
  index=False
)

---
x 만들기